In [ ]:
import math
import mitsuba as mi
import drjit as dr
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
import rawpy

# Set Mitsuba variant for differentiable rendering.
mi.set_variant("cuda_ad_rgb")

def cr2_to_exr(cr2_path: str, exr_path: str) -> None:
    """
    Convert a CR2 RAW image to an EXR file.

    Parameters:
        cr2_path (str): Path to the input CR2 file.
        exr_path (str): Path to save the output EXR file.
    """
    # Load the CR2 file using rawpy.
    with rawpy.imread(cr2_path) as raw:
        # Post-process the RAW image.
        rgb = raw.postprocess(no_auto_bright=True, output_bps=16)
    
    # Convert to float32 and normalize pixel values to [0, 1].
    rgb_float = np.float32(rgb) / 65535.0

    # Write the image as an EXR file using OpenCV.
    if not cv2.imwrite(exr_path, rgb_float):
        raise IOError(f"Failed to write EXR file to {exr_path}")
    print(f"Successfully converted '{cr2_path}' to '{exr_path}'.")

cr2_to_exr("IMG_0001.CR2", "observed.exr")

# Load the reference observed image (converted from your CR2)
# Here we assume "observed.exr" is a high-quality rendering of the scene.
observed_bitmap = mi.Bitmap("observed.exr").convert(mi.Bitmap.PixelFormat.RGB, mi.Struct.Type.Float32)
observed_np = observed_bitmap.numpy()  # shape (H, W, 3)
observed_torch = torch.tensor(observed_np, dtype=torch.float32, device="cuda")

# Get image dimensions.
H, W, _ = observed_np.shape

# Define differentiable parameters for light direction (spherical coordinates).
# Theta (angle from z-axis) and phi (angle in x-y plane)
theta = torch.nn.Parameter(torch.tensor(math.pi/4, device="cuda"))  # initial guess: 45°
phi   = torch.nn.Parameter(torch.tensor(math.pi/4, device="cuda"))    # initial guess: 45°

optimizer = torch.optim.Adam([theta, phi], lr=0.01)

def create_scene_dict(theta_val, phi_val):
    """
    Create a Mitsuba scene dictionary using the current light direction defined
    by theta and phi. The scene includes a simple sphere, a perspective camera, and
    a distant emitter whose direction is determined by theta and phi.
    """
    # Convert spherical to Cartesian coordinates.
    # x = sin(theta)*cos(phi), y = sin(theta)*sin(phi), z = cos(theta)
    x = math.sin(theta_val) * math.cos(phi_val)
    y = math.sin(theta_val) * math.sin(phi_val)
    z = math.cos(theta_val)
    # For a distant light, place the emitter far away in the light's direction.
    light_origin = [x * 100, y * 100, z * 100]
    # Use Mitsuba's look_at to build the emitter transform.
    emitter_transform = mi.ScalarTransform4f.look_at(
        origin=light_origin,
        target=[0, 0, 0],
        up=[0, 0, 1]
    )
    scene = {
        "type": "scene",
        "integrator": {"type": "path"},
        "sensor": {
            "type": "perspective",
            "fov": 45,
            "to_world": mi.ScalarTransform4f.look_at(
                origin=[0, 0, 10],
                target=[0, 0, 0],
                up=[0, 1, 0]
            ),
            "film": {
                "type": "hdrfilm",
                "width": W,
                "height": H,
                "file_format": "openexr",
                "pixel_format": "rgb"
            },
            "sampler": {"type": "independent", "sample_count": 16}
        },
        # Use a "distant" emitter (directional light).
        "emitter": {
            "type": "distant",
            "to_world": emitter_transform,
        },
        # Simple geometry: a sphere at the origin.
        "shape": {
            "type": "sphere",
            "center": [0, 0, 0],
            "radius": 1.0,
            "bsdf": {"type": "diffuse", "reflectance": {"type": "rgb", "value": [0.8, 0.8, 0.8]}}
        }
    }
    return scene

def render_scene(theta_val, phi_val):
    """
    Given the current light direction parameters (theta, phi), create the scene,
    render it with Mitsuba, and return the rendered image as a torch tensor.
    """
    scene_dict = create_scene_dict(theta_val, phi_val)
    scene = mi.load_dict(scene_dict)
    img = mi.render(scene)
    bmp = mi.Bitmap(img)
    rendered_np = bmp.numpy()  # shape (H, W, 3) in float32
    # Convert the numpy array to torch tensor.
    rendered_torch = torch.tensor(rendered_np, dtype=torch.float32, device="cuda")
    return rendered_torch

# Define the loss function: L2 loss between rendered and observed images.
def loss_fn():
    rendered = render_scene(theta.item(), phi.item())
    loss = torch.nn.functional.mse_loss(rendered, observed_torch)
    return loss

# Optimization loop.
num_iterations = 200
for i in range(num_iterations):
    optimizer.zero_grad()
    loss = loss_fn()
    loss.backward()
    optimizer.step()
    if (i+1) % 10 == 0:
        print(f"Iteration {i+1}: Loss = {loss.item():.6f}, Theta = {theta.item():.4f}, Phi = {phi.item():.4f}")

# Final estimated light direction.
final_theta = theta.item()
final_phi = phi.item()
x = math.sin(final_theta) * math.cos(final_phi)
y = math.sin(final_theta) * math.sin(final_phi)
z = math.cos(final_theta)
print("Estimated light direction (Cartesian):", (x, y, z))

# Optionally, visualize the final rendered image.
final_rendered = render_scene(final_theta, final_phi).cpu().detach().numpy()
plt.imshow(final_rendered / final_rendered.max())
plt.title("Rendered Image with Optimized Light Direction")
plt.axis("off")
plt.show()


ModuleNotFoundError: No module named 'mitsuba'